In [9]:
import os
import random
import evaluate
from dotenv import load_dotenv
from datasets import load_dataset, DatasetDict, Audio
from transformers import (
    WhisperProcessor, 
    WhisperForConditionalGeneration,
)
from typing import *


if os.path.exists(".env"):
    load_dotenv()
else:
    load_dotenv("env.env")

WHISPER_MODEL = "openai/whisper-medium"
MODEL_OUTPUT_DIR = "model-" + WHISPER_MODEL.replace("/", "-")
LOGS_OUTPUT_DIR = "logs-" + WHISPER_MODEL.replace("/", "-")
LANGUAGE = "Arabic"
LANGUAGE_TAG = "ar"
TASK = "transcribe"
WER_EVALUATION_METRIC = "wer"
CER_EVALUATION_METRIC = "cer"
AUDIO_SAMPLE_RATE = 16000
CONFIG_PATH = "./config.yaml"
AUTH_TOKEN = os.getenv("HUGGINGFACE_ACCESS_TOKEN")

# notebook_login()

In [10]:
common_voice = DatasetDict()
common_voice["test"] = load_dataset(
    "mozilla-foundation/common_voice_11_0",
    LANGUAGE_TAG, 
    split="test", 
    use_auth_token=AUTH_TOKEN
)
common_voice = common_voice.remove_columns(
    [
        "accent", 
        "age", 
        "client_id", 
        "down_votes", 
        "gender", 
        "locale", 
        "path", 
        "segment", 
        "up_votes"
    ]
)
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=AUDIO_SAMPLE_RATE))
common_voice

Found cached dataset common_voice_11_0 (/root/.cache/huggingface/datasets/mozilla-foundation___common_voice_11_0/ab/11.0.0/3f27acf10f303eac5b6fbbbe02495aeddb46ecffdb0a2fe3507fcfbf89094631)


DatasetDict({
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 9124
    })
})

In [11]:
wer_metric = evaluate.load(WER_EVALUATION_METRIC)
cer_metric = evaluate.load(CER_EVALUATION_METRIC)

In [12]:
def evaluate_sample(model, processor, sample, wer_metric, cer_metric):
    input_features = processor(sample["audio"]["array"], sampling_rate=sample["audio"]["sampling_rate"], return_tensors="pt")
    generated_ids = model.generate(**input_features)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    reference_text = sample["sentence"]
    wer =  100 * wer_metric.compute(predictions=[generated_text], references=[reference_text])
    cer =  100 * cer_metric.compute(predictions=[generated_text], references=[reference_text])
    return generated_text, wer, cer

## Base Whisper Performance on Arabic

In [13]:
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL, language=LANGUAGE, task=TASK)
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL)

model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

all_wer = []
all_cer = []
rand_sample_idxs = [random.randint(0, len(common_voice["test"])) for i in range(0, 10)]
for i in rand_sample_idxs:
    sample = common_voice["test"][i]
    generated_text, wer, cer = evaluate_sample(model, processor, sample, wer_metric, cer_metric)
    all_wer.append(wer)
    all_cer.append(cer)
    print(f"Generated Text: {generated_text}")
    print(f"Reference Text: {sample['sentence']}")
    print(f"WER: {wer}\t CER: {cer}")
    print("-"*50)
    
average_wer = sum(all_wer) / len(all_wer)
average_cer = sum(all_cer) / len(all_cer)
print(f"Average WER: {average_wer} \tAverage: {average_cer}")


Generated Text:  Sus died, Sarge.
Reference Text: Дсазҵааит саргьы.
WER: 150.0	 CER: 88.23529411764706
--------------------------------------------------
Generated Text:  We will be back.
Reference Text: Уи игәы ҭгәырӷьаауа иаб ихәда инапқәа накәиршеит.
WER: 100.0	 CER: 91.83673469387756
--------------------------------------------------
Generated Text:  and then I'll put the apple on top.
Reference Text: Иҭыҵыз ашәҟәы аҟнытә иаагоуп.
WER: 200.0	 CER: 110.34482758620689
--------------------------------------------------
Generated Text:  Timur, Wanlah Shadugukhshu, Nan, ushpaghuh
Reference Text: Ҭемыр, уан лаҳәшьа дукәыхшоуп, нан, ушԥаҟоу?
WER: 100.0	 CER: 88.63636363636364
--------------------------------------------------
Generated Text:  Sergey Gosslitis data as narad salab.
Reference Text: Саргьы гослитиздатаа снарыдҵаалап.
WER: 200.0	 CER: 100.0
--------------------------------------------------
Generated Text:  I'm not sure if I can do this, but I'm sure I can.
Reference Text: Лад

## FineTuned Whisper Performance on Arabic

In [14]:
finetuned_model = WhisperForConditionalGeneration.from_pretrained(f"{MODEL_OUTPUT_DIR}/checkpoint-5000")

finetuned_model.generation_config.language = LANGUAGE
finetuned_model.generation_config.task = TASK
finetuned_model.generation_config.forced_decoder_ids = None

all_wer = []
all_cer = []

for i in rand_sample_idxs:
    sample = common_voice["test"][i]
    generated_text, wer, cer = evaluate_sample(finetuned_model, processor, sample, wer_metric, cer_metric)
    all_wer.append(wer)
    all_cer.append(cer)
    print(f"Generated Text: {generated_text}")
    print(f"Reference Text: {sample['sentence']}")
    print(f"WER: {wer}\t CER: {cer}")
    print("-"*50)
    
average_wer = sum(all_wer) / len(all_wer)
average_cer = sum(all_cer) / len(all_cer)
print(f"Average WER: {average_wer} \tAverage: {average_cer}")


Generated Text:  Дсазҵааит саргьы.
Reference Text: Дсазҵааит саргьы.
WER: 0.0	 CER: 0.0
--------------------------------------------------
Generated Text:  Уи игәыдгәырӷьаауа иаб ихәда инапы акәиршеит.
Reference Text: Уи игәы ҭгәырӷьаауа иаб ихәда инапқәа накәиршеит.
WER: 57.14285714285714	 CER: 12.244897959183673
--------------------------------------------------
Generated Text: Иҭыҵыз ажәҟәы аҟнытә иаагоуп.
Reference Text: Иҭыҵыз ашәҟәы аҟнытә иаагоуп.
WER: 25.0	 CER: 3.4482758620689653
--------------------------------------------------
Generated Text: Ҭымыр, уан лаҳәшьа дукәыхшоуп, нан, ушпаҟоу?
Reference Text: Ҭемыр, уан лаҳәшьа дукәыхшоуп, нан, ушԥаҟоу?
WER: 33.33333333333333	 CER: 4.545454545454546
--------------------------------------------------
Generated Text: Саргьы гәыслит изда дааснарыдҵаалап.
Reference Text: Саргьы гослитиздатаа снарыдҵаалап.
WER: 100.0	 CER: 17.647058823529413
--------------------------------------------------
Generated Text: Лад ахаҳқәа данрыбжьыԥоз иақ